In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [2]:
base_dir = Path.cwd()
OUTPUT_DIR = base_dir / "output"
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-dnam-40k")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.dnam.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

RANDOM_SEED = 42
N_JOBS = 4
N_SPLITS = 5

from helpers import run_cv, analyze_feature_importances

In [3]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train = X_train.astype(np.float32)

y_train = pd.read_csv(y_train_path, index_col=0)
X_train, y_train

(              cg06745684  cg16034168  cg08166720  cg26992245  cg11161853  \
 id                                                                         
 ALL_1277          0.6023      0.9639      0.9205      0.8448      0.0287   
 ALL_1286          0.7584      0.9572      0.9278      0.8573      0.0294   
 ALL_1300          0.8942      0.9654      0.9193      0.8789      0.0269   
 ALL_1322          0.8010      0.9525      0.9410      0.6207      0.0257   
 ALL_1352          0.3665      0.9360      0.9238      0.8766      0.6753   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2991      0.8754      0.9817      0.9702      0.9372      0.0225   
 TCGA-AB-2992      0.8416      0.9771      0.9639      0.9356      0.0376   
 TCGA-AB-2997      0.5491      0.9818      0.9713      0.9561      0.0288   
 TCGA-AB-3007      0.8527      0.9794      0.9619      0.9402      0.0230   
 TCGA-AB-3012      0.9036      0.9791      0.9684      0.9396      0.0254   

In [4]:
# Show class distributions in y_train and y_test
print("y_train class distribution:")
print(y_train.value_counts())

y_train class distribution:
class                   
AML, KMT2A-r                241
B-ALL, high hyperdiploid    229
B-ALL, ETV6::RUNX1          199
AML, RUNX1::RUNX1T1         174
AML, NPM1alt                130
AML, CBFB::MYH11            127
Non-leukemic                 95
T-ALL                        73
AML, NUP98-r                 69
AML, CEBPAalt                55
B-ALL, PAX5alt               47
B-ALL, KMT2A-r               43
B-ALL, TCF3::PBX1            26
B-ALL, DUX4-r                22
AML, CBFA2T3::GLIS2          22
AML, DEK::NUP214             19
B-ALL, BCR::ABL1             17
AML, APL-PML::RARA           16
AML, ETV6-r                  16
B-ALL, ETV6::RUNX1-like      15
B-ALL, iAMP21                15
B-ALL, ZNF384-r              15
B-ALL, BCR::ABL1-like        14
AML, KAT6A-r                  8
AML, PICALM::MLLT10           6
AML, FUS::ERG                 6
B-ALL, MEF2D-r                6
B-ALL, PAX5 P80R              6
Name: count, dtype: int64


In [5]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_train

array(['B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', ..., 'AML, APL-PML::RARA',
       'AML, APL-PML::RARA', 'AML, APL-PML::RARA'],
      shape=(1711,), dtype=object)

In [6]:
# Read in reduced feature set from previous step
reduced_features_path = os.path.join(OUTPUT_DIR, "selected_features.2.txt")
with open(reduced_features_path, 'r') as f:
    selected_features = [line.strip() for line in f]

# Subset X_train to only include selected features
X_train_reduced = X_train[selected_features]
X_train_reduced.shape

(1711, 17343)

In [7]:
# Run cross-validation and analyze feature importances
cv_results = run_cv(X_train_reduced, y_train, n_jobs=10)
cv_results

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[CV] END .................................................... total time=12.9min
[CV] END .................................................... total time=13.1min
[CV] END .................................................... total time=13.3min
[CV] END .................................................... total time=13.4min
[CV] END .................................................... total time=13.4min
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.841 ± 0.021
Gap:            0.159


[Parallel(n_jobs=10)]: Done   5 out of   5 | elapsed: 13.4min finished


{'fit_time': array([802.17042232, 785.64957094, 772.04099774, 801.26448989,
        796.15830183]),
 'score_time': array([0.07865167, 0.07693672, 0.08012223, 0.07772899, 0.08053422]),
 'estimator': [RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42)],
 'test_score': array([0.8454158 , 0.87443888, 0.84336532, 0.80931185, 0.83160988]),
 'train_score': array([1., 1., 1., 1.,

In [8]:
# Analyze feature importances and save results
importance_df, freq_df, selected_features = analyze_feature_importances(OUTPUT_DIR, cv_results, X_train_reduced, suffix="3", freq_threshold=1)

Selected 4820 features with frequency >= 1. Saved to /home/mariya/Development/pallas/training/dnam/output/selected_features.3.txt.
